# Europa Waters LLC: Solar Filament CV V3
## YOLO11s instance segmentation + U-Net crop refinement

**Run and submit this notebook yourself. Nothing launches from this repository.**
V1 public PQ: 0.29. Focused V2 public PQ: 0.32. The offline disjoint-mask-repaired
V3 submission 58316663 achieved public PQ 0.35 (API-verified complete).
The original overlapping-mask submission was rejected. This updated pipeline
recalibrates with disjoint masks; it is not identical to offline repair and
has no separately measured leaderboard result. No reproduction of the
reference notebook's reported ~0.69 score is promised.

Motivation: our previous semantic U-Net identifies foreground and separates
instances using connected components. YOLO segmentation instead predicts
individual instance masks; a crop U-Net may recover fine local boundaries.
The refiner cannot recover filaments the detector never proposes.

This notebook is an independent implementation informed by the supplied
YOLO+crop-refinement notebook. It uses the maintained Segmentation Models
PyTorch U-Net rather than copying its custom network. No third-party source
is relicensed as our own. Ultralytics has separate AGPL/enterprise terms;
review those terms before commercial redistribution or deployment.

**Differences from the supplied reference:** observation-day grouping rather
than random filename splitting; separate annotator training targets rather
than selecting the annotator with most instances; one foreground class
rather than chirality classes; consistent COCO rasterization for crop targets;
float32 loss reductions; tuning-only stage/threshold selection; explicit
checkpoint, zero-detection, and RLE validation. These improve methodology
and reliability, but may change results. No score parity is claimed.

Defaults: 1024px YOLO11s-seg, batch 4, **up to 60 epochs**, patience 8,
75-minute detector training allowance. Crop refiner: 256px crops with
1.8x context, ResNet18 encoder, batch 16, **up to 20 epochs**, patience 5,
30-minute allowance. Preparation, downloads, calibration, and inference are
additional time; this is NOT a hard two-hour end-to-end limit.
Both stages can finish early or truncate before convergence.
No CLAHE by default: first test the instance-aware architectural change
without stacking further contrast transformations.

Use a fresh Kaggle notebook with competition data, Internet, and T4 GPU.
Single-GPU training avoids notebook multi-process/DDP launch complications.
No automatic OOM fallback changes batch or resolution. Set batch=2 with a
new run name if your GPU cannot fit batch=4.

In [ ]:
%pip install -q 'ultralytics>=8.3,<9' 'segmentation-models-pytorch>=0.3.3,<0.7' pycocotools scipy 'opencv-python-headless>=4.9,<5' PyYAML

In [ ]:
import csv
import hashlib
import importlib.metadata
import itertools
import json
import random
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage as ndi
from scipy.optimize import linear_sum_assignment
from PIL import Image
import pycocotools.mask as mask_utils
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

CFG = dict(seed=2026, image_size=512, batch_size=8, epochs=20,
           learning_rate=3e-4, encoder='resnet18', architecture='Unet',
           pretrained=True, workers=0, tune_fraction=0.15, holdout_fraction=0.15,
           tune_image_limit=32, thresholds=[0.35, 0.5, 0.65],
           min_areas=[64, 256], contrast_thresholds=[0.025, 0.04, 0.06])
RUN_NAME = 'v3_yolo11s_crop_unet_seed2026'
RUN_TRAINING = False
RUN_CALIBRATION = False
SMOKE_TEST = False
SUBMIT_TO_KAGGLE = False
DATA_ROOT_OVERRIDE = None
CHECKPOINT_INPUT_DIR = None  # Path to an attached experiment folder for inference-only reproduction.
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
              else Path.cwd() / 'cv_outputs') / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path(CHECKPOINT_INPUT_DIR) if CHECKPOINT_INPUT_DIR else OUTPUT_DIR
CHECKPOINT_PATH = INPUT_DIR / 'best_model.pt'
COMPETITION = 'filament-segmentation-2026'
DATASET_NAME = 'MAGFiLO_1.0_Kaggle_2026'
ANNOTATION_NAME = 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
FULL_SHAPE = (2048, 2048)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    print('WARNING: CPU selected. Enable a Kaggle GPU before full training.')
if CFG['image_size'] % 32 or CFG['batch_size'] < 2 or CFG['epochs'] < 1:
    raise ValueError('Use image_size divisible by 32, batch_size >= 2, and epochs >= 1.')
print('Device:', DEVICE, 'outputs:', OUTPUT_DIR)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False))


In [ ]:
candidates = ([Path(DATA_ROOT_OVERRIDE)] if DATA_ROOT_OVERRIDE else [
    Path('/kaggle/input/competitions') / COMPETITION / DATASET_NAME,
    Path('/kaggle/input') / COMPETITION / DATASET_NAME,
    Path.cwd() / 'data' / DATASET_NAME, Path.cwd() / DATASET_NAME])
if DATA_ROOT_OVERRIDE is None and Path('/kaggle/input').is_dir():
    candidates += sorted(Path('/kaggle/input').glob(f'*/{DATASET_NAME}'))
    candidates += sorted(Path('/kaggle/input').glob(f'*/*/{DATASET_NAME}'))
valid_roots = [p for p in candidates if (p / 'train' / ANNOTATION_NAME).is_file()
               and (p / 'test' / 'test_images').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Attach the competition data or set DATA_ROOT_OVERRIDE to its dataset root.')
DATA_ROOT = valid_roots[0]
train_paths = sorted((DATA_ROOT / 'train' / 'train_images').glob('*.jpeg'))
test_paths = sorted((DATA_ROOT / 'test' / 'test_images').glob('*.jpeg'))
if not train_paths or not test_paths:
    raise FileNotFoundError('Train and test must both contain JPEG files.')
with (DATA_ROOT / 'train' / ANNOTATION_NAME).open() as stream:
    coco = json.load(stream)
records = coco['images']
if len({r['id'] for r in records}) != len(records):
    raise ValueError('Duplicate COCO image IDs.')
records_by_name = defaultdict(list)
annotations_by_id = defaultdict(list)
for record in records:
    if (record['height'], record['width']) != FULL_SHAPE:
        raise ValueError(f'Unexpected dimensions for {record["id"]}')
    records_by_name[record['file_name']].append(record)
for ann in coco['annotations']:
    if ann.get('iscrowd', 0) != 0:
        raise ValueError('Local PQ assumes individual non-crowd filaments.')
    annotations_by_id[ann['image_id']].append(ann)
known_ids = {r['id'] for r in records}
if set(annotations_by_id) - known_ids:
    raise ValueError('Orphan annotation image IDs.')
if len({a['id'] for a in coco['annotations']}) != len(coco['annotations']):
    raise ValueError('Duplicate filament annotation IDs.')
train_lookup = {p.name: p for p in train_paths}
if set(records_by_name) - set(train_lookup):
    raise FileNotFoundError('Annotations reference missing training files.')
if set(train_lookup) & {p.name for p in test_paths}:
    raise ValueError('Train/test filenames overlap.')
days = sorted({name[:8] for name in records_by_name})
if len(days) < 6:
    raise ValueError('Not enough observing days for three nonempty grouped splits.')
random.Random(CFG['seed']).shuffle(days)
n_tune = max(1, round(len(days) * CFG['tune_fraction']))
n_holdout = max(1, round(len(days) * CFG['holdout_fraction']))
if n_tune + n_holdout >= len(days):
    raise ValueError('Tuning and holdout fractions leave no training days.')
day_sets = dict(tune=set(days[:n_tune]), holdout=set(days[n_tune:n_tune+n_holdout]),
                train=set(days[n_tune+n_holdout:]))
split_names = {split: sorted(n for n in records_by_name if n[:8] in selected)
               for split, selected in day_sets.items()}
split_records = {split: [r for name in names for r in records_by_name[name]]
                 for split, names in split_names.items()}
for first, second in itertools.combinations(day_sets, 2):
    assert not day_sets[first] & day_sets[second]
annotation_sha256 = hashlib.sha256((DATA_ROOT / 'train' / ANNOTATION_NAME).read_bytes()).hexdigest()
manifest = dict(seed=CFG['seed'], grouping='observation-day', annotation_sha256=annotation_sha256,
                policy='separate annotator targets; union within each target', splits=split_names)
write_json(OUTPUT_DIR / 'split_manifest.json', manifest)
print('Data root:', DATA_ROOT)
print('JPEGs:', len(train_paths), 'train,', len(test_paths), 'test')
print('Annotation sets:', len(records), 'filaments:', len(coco['annotations']))
print('Categories:', Counter(a['category_id'] for a in coco['annotations']))
print('Splits:', {s: (len(split_names[s]), len(split_records[s])) for s in split_names})


In [ ]:
@lru_cache(maxsize=128)
def small_image(filename):
    with Image.open(train_lookup[filename]) as image:
        if image.mode != 'L' or image.size != (2048, 2048):
            raise ValueError(f'{filename}: expected 2048 x 2048 grayscale JPEG.')
        image = image.resize((CFG['image_size'], CFG['image_size']), Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.float32) / 255.0

def ann_rle(annotation):
    segmentation = annotation['segmentation']
    if isinstance(segmentation, list):
        if not segmentation or any(len(p) < 6 or len(p) % 2 for p in segmentation):
            raise ValueError(f'Invalid polygon: {annotation["id"]}')
        rle = mask_utils.merge(mask_utils.frPyObjects(segmentation, *FULL_SHAPE))
    elif isinstance(segmentation, dict):
        if list(segmentation['size']) != list(FULL_SHAPE):
            raise ValueError('RLE dimensions do not match the image.')
        if isinstance(segmentation['counts'], list):
            rle = mask_utils.frPyObjects(segmentation, *FULL_SHAPE)
        else:
            counts = segmentation['counts']
            rle = dict(size=list(FULL_SHAPE), counts=counts.encode('ascii') if isinstance(counts, str) else counts)
    else:
        raise TypeError(f'Unsupported segmentation: {annotation["id"]}')
    if mask_utils.area(rle) <= 0:
        raise ValueError(f'Empty ground-truth filament: {annotation["id"]}')
    return rle

@lru_cache(maxsize=512)
def gt_rles(image_id):
    return [ann_rle(a) for a in annotations_by_id[image_id]]

def union_mask(rles):
    if not rles:
        return np.zeros(FULL_SHAPE, dtype=np.uint8)
    return mask_utils.decode(mask_utils.merge(rles))

@lru_cache(maxsize=128)
def small_target(image_id):
    full_mask = union_mask(gt_rles(image_id))
    resized = Image.fromarray(full_mask).resize((CFG['image_size'], CFG['image_size']), Image.Resampling.NEAREST)
    return np.asarray(resized, dtype=np.float32)

class FilamentDataset(Dataset):
    def __init__(self, entries, augment=False):
        self.entries = entries
        self.augment = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):
        record = self.entries[index]
        image = small_image(record['file_name'])
        target = small_target(record['id'])
        if self.augment:
            turns = random.randrange(4)
            image, target = np.rot90(image, turns), np.rot90(target, turns)
            if random.random() < 0.5:
                image, target = np.fliplr(image), np.fliplr(target)
        x = torch.from_numpy(np.ascontiguousarray((image - 0.5) / 0.5))[None]
        y = torch.from_numpy(np.ascontiguousarray(target))[None]
        return x, y



In [ ]:
def build_model(config, pretrained=False):
    constructors = {'Unet': smp.Unet, 'FPN': smp.FPN}
    if config['architecture'] not in constructors:
        raise ValueError('Supported architectures: Unet, FPN.')
    return constructors[config['architecture']](encoder_name=config['encoder'],
        encoder_weights='imagenet' if pretrained else None, in_channels=1, classes=1, activation=None)

def segmentation_loss(logits, target):
    probabilities = logits.sigmoid()
    intersection = (probabilities * target).sum(dim=(1, 2, 3))
    denominator = probabilities.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = 1 - ((2 * intersection + 1) / (denominator + 1)).mean()
    return 0.5 * F.binary_cross_entropy_with_logits(logits, target) + 0.5 * dice



In [ ]:
@torch.inference_mode()
def predict_probability(path):
    with Image.open(path) as image:
        if image.mode != 'L' or image.size != (2048, 2048):
            raise ValueError(f'{path.name}: invalid image mode or dimensions.')
        image = image.resize((CFG['image_size'], CFG['image_size']), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(np.ascontiguousarray((array - 0.5) / 0.5))[None, None].to(DEVICE)
    probability = model(tensor).sigmoid()[0, 0].cpu().numpy()
    if not np.isfinite(probability).all():
        raise FloatingPointError(f'Nonfinite prediction: {path.name}')
    return probability

def full_probability(probability):
    resized = Image.fromarray(np.asarray(probability, dtype=np.float32)).resize(
        (2048, 2048), Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32)

def contrast_foreground(filename, threshold):
    image = small_image(filename)
    disk = ndi.binary_erosion(image > 0.08, iterations=max(1, CFG['image_size']//128))
    background = ndi.gaussian_filter(image, sigma=CFG['image_size']/32)
    return disk & ((background - image) > threshold)

def instance_rles(probability, threshold, min_area):
    if min_area < 1 or not 0 <= threshold <= 1:
        raise ValueError('Invalid threshold or minimum area.')
    binary = full_probability(probability) >= threshold
    labels, count = ndi.label(binary, structure=np.ones((3, 3), dtype=np.uint8))
    areas = np.bincount(labels.ravel())
    result = []
    for label in range(1, count+1):
        if areas[label] >= min_area:
            mask = np.asfortranarray(labels == label, dtype=np.uint8)
            result.append(mask_utils.encode(mask))
    return result

def local_scores(predicted, truth):
    n_pred, n_true = len(predicted), len(truth)
    overlaps = (mask_utils.iou(predicted, truth, [0]*n_true)
                if n_pred and n_true else np.zeros((n_pred, n_true)))
    matched = []
    if n_pred and n_true:
        valid = overlaps > 0.5
        # Maximum cardinality first, then overlap; greedy matching can lose true positives.
        rewards = valid * (1 + overlaps / (min(n_pred, n_true) + 1))
        rows, columns = linear_sum_assignment(-rewards)
        matched = [float(overlaps[r, c]) for r, c in zip(rows, columns) if valid[r, c]]
    tp = len(matched); fp = n_pred-tp; fn = n_true-tp
    denominator = tp + 0.5*fp + 0.5*fn
    pq = sum(matched)/denominator if denominator else 1.0
    pred_union, true_union = union_mask(predicted).astype(bool), union_mask(truth).astype(bool)
    intersection = np.count_nonzero(pred_union & true_union)
    union = np.count_nonzero(pred_union | true_union)
    pixels = np.count_nonzero(pred_union) + np.count_nonzero(true_union)
    return dict(pq=pq, dice=2*intersection/pixels if pixels else 1.0,
                iou=intersection/union if union else 1.0, tp=tp, fp=fp, fn=fn,
                split_relations=int(np.count_nonzero((overlaps > 0.1).sum(axis=0) > 1)),
                merge_relations=int(np.count_nonzero((overlaps > 0.1).sum(axis=1) > 1)))

def evaluate(names, parameters, probability_cache=None):
    rows = []
    for filename in names:
        if parameters['method'] == 'unet':
            probability = (probability_cache[filename] if probability_cache is not None
                           else predict_probability(train_lookup[filename]))
            threshold = parameters['threshold']
        elif parameters['method'] == 'contrast':
            probability = contrast_foreground(filename, parameters['contrast_threshold']).astype(np.float32)
            threshold = 0.5
        else:
            raise ValueError('Unknown segmentation method.')
        predicted = instance_rles(probability, threshold, parameters['min_area'])
        for record in records_by_name[filename]:
            rows.append(dict(filename=filename, image_id=record['id'],
                             **local_scores(predicted, gt_rles(record['id']))))
    if not rows:
        raise ValueError('Evaluation set is empty.')
    # Average annotators per image, then physical images; do not overweight triply labeled images.
    grouped = defaultdict(list)
    for row in rows: grouped[row['filename']].append(row)
    metrics = {key: float(np.mean([np.mean([r[key] for r in group]) for group in grouped.values()]))
               for key in ['pq', 'dice', 'iou', 'tp', 'fp', 'fn', 'split_relations', 'merge_relations']}
    return metrics, rows


## Required disjoint instance masks
The first V3 submission was rejected because predicted masks overlapped.
Box NMS does not guarantee disjoint pixel masks, especially after crop refinement.
The same deterministic `first-mask-wins-v1` policy now runs during tuning,
error visualization, and export: retain proposal order, give contested pixels
to the earliest mask, and drop later empty or below-minimum-area remnants.
No new foreground pixels are created. The repair preserves order; it does
not claim optimal boundaries or infer confidence from a saved CSV.

The following cell embeds `submission_masks.py` from the repository so the
notebook remains standalone. Tests verify the embedded source stays identical.
Export independently rejects any remaining overlap after serialization.
Existing checkpoints remain usable, but old calibration metrics do not
describe this policy. Disable both V3 training flags and re-run calibration
before a new checkpoint-based export. Alternatively, the repository's offline
repair command can fix the saved CSV without training or GPU inference.

In [ ]:
"""Disjoint instance masks and an offline repair command for saved V3 CSVs."""

import argparse
import csv
import hashlib
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import pycocotools.mask as mask_utils


MASK_POLICY = "first-mask-wins-v1"
SUBMISSION_FIELDS = ["filament_id", "segmentation_rle"]


def decode_instance(rle, shape):
    if tuple(rle["size"]) != tuple(shape):
        raise ValueError(f"Instance shape {rle['size']} does not match {shape}.")
    mask = mask_utils.decode(rle)
    if mask.shape != tuple(shape) or not np.isin(mask, [0, 1]).all() or not mask.any():
        raise ValueError("An instance must be a nonempty binary mask.")
    return mask.astype(bool)


def disjoint_instances(instances, shape=(2048, 2048), min_area=1):
    """Keep earlier masks unchanged; later masks may claim only unowned pixels."""
    if min_area < 1:
        raise ValueError("min_area must be positive.")
    occupied = np.zeros(shape, dtype=bool)
    result, kept_indices, dropped_indices = [], [], []
    overlap_pixels = 0
    for index, rle in enumerate(instances):
        mask = decode_instance(rle, shape)
        overlap_pixels += int(np.count_nonzero(mask & occupied))
        mask &= ~occupied
        if np.count_nonzero(mask) < min_area:
            dropped_indices.append(index)
            continue
        occupied |= mask
        result.append(mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))))
        kept_indices.append(index)
    return result, dict(
        policy=MASK_POLICY, overlap_pixels_removed=overlap_pixels,
        kept_indices=kept_indices, dropped_indices=dropped_indices,
    )


def validate_disjoint(instances, shape=(2048, 2048)):
    occupied = np.zeros(shape, dtype=bool)
    for index, rle in enumerate(instances):
        mask = decode_instance(rle, shape)
        if np.any(occupied & mask):
            raise ValueError(f"Instance {index} overlaps an earlier instance.")
        occupied |= mask


def submission_groups(rows, test_stems, shape):
    groups, seen = defaultdict(list), set()
    for row in rows:
        if set(row) != set(SUBMISSION_FIELDS):
            raise ValueError("CSV must have exactly the two competition columns.")
        identity = row["filament_id"]
        stem, separator, number = identity.rpartition("_")
        if (not separator or not number.isdigit() or int(number) < 1
                or stem not in test_stems or identity in seen):
            raise ValueError(f"Unknown, invalid, or duplicate filament_id: {identity}")
        seen.add(identity)
        rle = dict(size=list(shape), counts=row["segmentation_rle"].encode("ascii"))
        groups[stem].append((row, rle))
    if not groups:
        raise ValueError("Submission has no predicted instances.")
    return groups


def validate_submission_rows(rows, test_stems, shape=(2048, 2048)):
    groups = submission_groups(rows, test_stems, shape)
    for stem, entries in groups.items():
        try:
            validate_disjoint([rle for _, rle in entries], shape)
        except ValueError as error:
            raise ValueError(f"{stem}: {error}") from error
    return dict(
        rows=len(rows), images_with_predictions=len(groups),
        zero_detection_images=sorted(set(test_stems) - set(groups)),
        overlapping_pixels=0,
    )


def repair_submission(source, destination, test_stems, min_area=20, shape=(2048, 2048)):
    source, destination = Path(source), Path(destination)
    audit_path = destination.with_suffix(".audit.json")
    if source.resolve() == destination.resolve():
        raise ValueError("Repair must not overwrite the original submission.")
    if destination.exists() or audit_path.exists():
        raise FileExistsError("Choose a new repair output path; existing evidence is preserved.")
    with source.open(newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != SUBMISSION_FIELDS:
            raise ValueError("Unexpected submission columns or column order.")
        original_rows = list(reader)
    groups = submission_groups(original_rows, test_stems, shape)
    repaired_by_id, image_audits = {}, []
    for stem, entries in groups.items():
        masks, audit = disjoint_instances([rle for _, rle in entries], shape, min_area)
        for index, rle in zip(audit["kept_indices"], masks):
            identity = entries[index][0]["filament_id"]
            repaired_by_id[identity] = dict(
                filament_id=identity, segmentation_rle=rle["counts"].decode("ascii"),
            )
        image_audits.append(dict(
            image=stem, overlap_pixels_removed=audit["overlap_pixels_removed"],
            input_instances=len(entries), output_instances=len(masks),
            dropped_ids=[entries[i][0]["filament_id"] for i in audit["dropped_indices"]],
        ))
    rows = [repaired_by_id[r["filament_id"]] for r in original_rows
            if r["filament_id"] in repaired_by_id]
    validation = validate_submission_rows(rows, test_stems, shape)
    temporary = destination.with_suffix(".tmp.csv")
    if temporary.exists():
        raise FileExistsError(f"Temporary file already exists: {temporary}")
    try:
        with temporary.open("x", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=SUBMISSION_FIELDS)
            writer.writeheader()
            writer.writerows(rows)
        with temporary.open(newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != SUBMISSION_FIELDS or list(reader) != rows:
                raise ValueError("Repaired CSV readback mismatch.")
        report = dict(
            policy=MASK_POLICY, min_area=min_area, validation=validation,
            input_csv=str(source), input_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),
            output_sha256=hashlib.sha256(temporary.read_bytes()).hexdigest(),
            input_rows=len(original_rows), dropped_rows=len(original_rows) - len(rows),
            images_with_overlap=sum(a["overlap_pixels_removed"] > 0 for a in image_audits),
            overlap_pixels_removed=sum(a["overlap_pixels_removed"] for a in image_audits),
            images=image_audits,
            note="Offline repair only. No retraining, recalibration, or submission. "
                 "Previous tuning scores do not measure these repaired masks.",
        )
        with audit_path.open("x") as stream:
            stream.write(json.dumps(report, indent=2, allow_nan=False))
        temporary.replace(destination)
    finally:
        if temporary.exists():
            temporary.unlink()
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("source", type=Path)
    parser.add_argument("destination", type=Path)
    parser.add_argument("--test-dir", type=Path, required=True)
    parser.add_argument("--min-area", type=int, default=20)
    args = parser.parse_args()
    test_stems = {p.stem for p in args.test_dir.rglob("*.jpeg")}
    if not test_stems:
        parser.error("--test-dir contains no test JPEGs.")
    report = repair_submission(args.source, args.destination, test_stems, args.min_area)
    print(json.dumps({key: value for key, value in report.items() if key != "images"}, indent=2))
    print("REPAIRED CSV:", args.destination, "No competition upload performed.")



In [ ]:
import gc
import yaml
from ultralytics import YOLO

V3 = dict(
    yolo_weights="yolo11s-seg.pt", image_size=1024, batch_size=4,
    yolo_epochs=60, yolo_patience=8, yolo_hours=1.25,
    crop_size=256, crop_context=1.8, minimum_crop=96,
    refiner_epochs=20, refiner_batch=16, refiner_lr=3e-4,
    refiner_patience=5, refiner_min_delta=1e-4, refiner_hours=0.5,
    tuning_images=32, confidences=[0.15, 0.25],
    refiner_thresholds=[0.5, 0.65], min_areas=[20, 64],
)
RUN_YOLO_TRAINING = True
RUN_REFINER_TRAINING = True
RUN_V3_CALIBRATION = True
RUN_V3_EXPORT = True
REFINER_PRETRAINED = True
ALLOW_CPU_SMOKE = False
YOLO_DEVICE = 0


def v3_seed():
    random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(CFG["seed"])


def v3_prepare_dataset():
    directory = OUTPUT_DIR / "yolo_dataset"
    split_manifest = dict(
        annotation_sha256=annotation_sha256, splits=split_names, settings=V3,
        smoke_test=SMOKE_TEST,
        annotation_policy="separate annotator targets; single foreground class",
    )
    saved_path = OUTPUT_DIR / "v3_plan.json"
    if saved_path.exists() and json.loads(saved_path.read_text()) != split_manifest:
        raise ValueError("V3 output has a different plan. Use a new RUN_NAME.")
    write_json(saved_path, split_manifest)
    for phase, label in [("train", "train"), ("tune", "val")]:
        image_dir, label_dir = directory / "images" / label, directory / "labels" / label
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)
        for index, record in enumerate(split_records[phase]):
            # Distinct label files for each annotator, but all copies remain in one day partition.
            stem = f'{Path(record["file_name"]).stem}__annotation{index}'
            destination = image_dir / f"{stem}.jpeg"
            source = train_lookup[record["file_name"]]
            if not destination.exists():
                destination.symlink_to(source.resolve())
            lines = []
            for annotation in annotations_by_id[record["id"]]:
                polygons = annotation["segmentation"]
                if not isinstance(polygons, list) or len(polygons) != 1:
                    raise ValueError("V3 YOLO conversion requires one polygon per instance; no silent geometry approximation.")
                points = np.asarray(polygons[0], dtype=np.float64)
                if points.size < 6 or points.size % 2 or not np.isfinite(points).all():
                    raise ValueError(f'Invalid polygon {annotation["id"]}')
                points = points.reshape(-1, 2)
                if np.any(points < 0) or np.any(points > 2048):
                    raise ValueError(f'Out-of-bounds polygon {annotation["id"]}')
                normalized = points / 2048
                lines.append("0 " + " ".join(f"{value:.8f}" for value in normalized.ravel()))
            (label_dir / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
    path = directory / "data.yaml"
    path.write_text(yaml.safe_dump(dict(
        path=str(directory.resolve()), train="images/train", val="images/val",
        names={0: "filament"},
    )))
    return path


def v3_train_yolo():
    if DEVICE.type != "cuda" and not ALLOW_CPU_SMOKE:
        raise RuntimeError("Enable a GPU before V3 full training.")
    data = v3_prepare_dataset()
    checkpoint = OUTPUT_DIR / "yolo_best.pt"
    if RUN_YOLO_TRAINING:
        if checkpoint.exists():
            raise FileExistsError("Saved YOLO checkpoint exists. Disable training to reuse or choose a new RUN_NAME.")
        v3_seed()
        detector = YOLO(V3["yolo_weights"])
        detector.train(
            data=str(data), epochs=V3["yolo_epochs"], imgsz=V3["image_size"],
            batch=V3["batch_size"], device=YOLO_DEVICE, workers=0,
            patience=V3["yolo_patience"], time=V3["yolo_hours"],
            seed=CFG["seed"], single_cls=True, mask_ratio=2, overlap_mask=False,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.3,
            fliplr=0.5, flipud=0.0, mosaic=0.3, degrees=10.0,
            scale=0.3, close_mosaic=5, cache=False,
            project=str(OUTPUT_DIR / "detector_runs"), name="yolo11s_seg",
            exist_ok=False, plots=False,
        )
        best = Path(detector.trainer.best)
        if not best.is_file():
            raise FileNotFoundError("YOLO did not produce best.pt.")
        shutil.copyfile(best, checkpoint)
        write_json(OUTPUT_DIR / "yolo_provenance.json", dict(
            sha256=hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
            settings=V3, annotation_sha256=annotation_sha256, smoke_test=SMOKE_TEST,
        ))
        del detector
        gc.collect()
    provenance = json.loads((OUTPUT_DIR / "yolo_provenance.json").read_text())
    if (provenance["sha256"] != hashlib.sha256(checkpoint.read_bytes()).hexdigest()
            or provenance["settings"] != V3 or provenance["smoke_test"] != SMOKE_TEST
            or provenance["annotation_sha256"] != annotation_sha256):
        raise ValueError("YOLO checkpoint provenance mismatch.")
    return YOLO(str(checkpoint))


def crop_bounds(mask):
    ys, xs = np.where(mask)
    if not len(xs):
        raise ValueError("Cannot crop an empty instance.")
    side = min(2048, max(V3["minimum_crop"], int(np.ceil(
        max(xs.max()-xs.min()+1, ys.max()-ys.min()+1) * V3["crop_context"]
    ))))
    x = int(np.clip(round((xs.max()+xs.min()+1-side)/2), 0, 2048-side))
    y = int(np.clip(round((ys.max()+ys.min()+1-side)/2), 0, 2048-side))
    return x, y, x+side, y+side


def crop_input(image, bounds):
    x0, y0, x1, y1 = bounds
    crop = image.crop((x0, y0, x1, y1)).resize(
        (V3["crop_size"], V3["crop_size"]), Image.Resampling.BILINEAR
    )
    array = np.asarray(crop, dtype=np.float32) / 255
    return np.repeat(((array - 0.5) / 0.5)[None], 3, axis=0).copy()


class InstanceCropDataset(Dataset):
    def __init__(self, records, augment=False):
        self.entries = [
            (record, annotation) for record in records
            for annotation in annotations_by_id[record["id"]]
        ]
        self.augment = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):
        record, annotation = self.entries[index]
        mask = mask_utils.decode(ann_rle(annotation))
        bounds = crop_bounds(mask)
        with Image.open(train_lookup[record["file_name"]]) as image:
            if image.mode != "L" or image.size != (2048, 2048):
                raise ValueError("Unexpected crop input image.")
            x = crop_input(image, bounds)
        x0, y0, x1, y1 = bounds
        target = Image.fromarray(mask[y0:y1, x0:x1]).resize(
            (V3["crop_size"], V3["crop_size"]), Image.Resampling.NEAREST
        )
        y = np.asarray(target, dtype=np.float32)[None]
        if self.augment:
            turns = random.randrange(4)
            x, y = np.rot90(x, turns, axes=(1, 2)), np.rot90(y, turns, axes=(1, 2))
            if random.random() < 0.5:
                x, y = np.flip(x, axis=2), np.flip(y, axis=2)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))


def v3_refiner(pretrained):
    return smp.Unet(
        encoder_name="resnet18", encoder_weights="imagenet" if pretrained else None,
        in_channels=3, classes=1, activation=None,
    ).to(DEVICE)


def v3_train_refiner():
    path = OUTPUT_DIR / "refiner_best.pt"
    if RUN_REFINER_TRAINING:
        if path.exists():
            raise FileExistsError("Saved refiner exists. Disable training or choose a new RUN_NAME.")
        v3_seed()
        refiner = v3_refiner(REFINER_PRETRAINED)
        datasets = {
            phase: InstanceCropDataset(split_records[phase], augment=phase == "train")
            for phase in ["train", "tune"]
        }
        loaders = {
            phase: DataLoader(
                dataset, batch_size=V3["refiner_batch"], shuffle=phase == "train",
                num_workers=0, drop_last=phase == "train" and len(dataset) % V3["refiner_batch"] == 1,
            ) for phase, dataset in datasets.items()
        }
        if any(len(loader) == 0 for loader in loaders.values()):
            raise ValueError("Empty crop dataset.")
        optimizer = torch.optim.AdamW(refiner.parameters(), lr=V3["refiner_lr"], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=V3["refiner_epochs"]
        )
        amp = DEVICE.type == "cuda"
        scaler = torch.amp.GradScaler("cuda", enabled=amp)
        best, significant_best, stale, history = float("inf"), float("inf"), 0, []
        deadline = time.monotonic() + V3["refiner_hours"] * 3600
        status = "max_epochs"
        for epoch in range(V3["refiner_epochs"]):
            totals = {}
            interrupted = False
            for phase, loader in loaders.items():
                refiner.train(phase == "train")
                total, count = 0.0, 0
                with torch.set_grad_enabled(phase == "train"):
                    for x, y in tqdm(loader, desc=f"Crop {epoch+1} {phase}"):
                        if time.monotonic() >= deadline:
                            interrupted = True
                            break
                        x, y = x.to(DEVICE), y.to(DEVICE)
                        optimizer.zero_grad(set_to_none=True)
                        with torch.autocast(device_type=DEVICE.type, enabled=amp):
                            logits = refiner(x)
                        loss = segmentation_loss(logits.float(), y.float())
                        if not torch.isfinite(loss):
                            raise FloatingPointError("Nonfinite crop loss.")
                        if phase == "train":
                            scaler.scale(loss).backward()
                            scaler.step(optimizer); scaler.update()
                        total += loss.item()*len(x); count += len(x)
                if interrupted:
                    break
                totals[phase] = total / count
            if interrupted:
                status = "budget_truncated"
                print("Crop budget expired; incomplete epoch discarded.")
                break
            history.append(dict(epoch=epoch+1, **totals))
            print("REFINER:", history[-1], flush=True)
            if totals["tune"] < best:
                best = totals["tune"]
                temporary = path.with_suffix(".tmp.pt")
                torch.save(dict(
                    state_dict={k: v.detach().cpu() for k, v in refiner.state_dict().items()},
                    epoch=epoch+1, settings=V3, annotation_sha256=annotation_sha256,
                    smoke_test=SMOKE_TEST,
                ), temporary)
                temporary.replace(path)
            if totals["tune"] < significant_best - V3["refiner_min_delta"]:
                significant_best, stale = totals["tune"], 0
            else:
                stale += 1
            write_json(OUTPUT_DIR / "refiner_history.json", history)
            scheduler.step()
            if stale >= V3["refiner_patience"]:
                status = "early_stopped"
                break
        write_json(OUTPUT_DIR / "refiner_runtime.json", dict(status=status, completed_epochs=len(history)))
        del optimizer, scheduler, scaler, refiner, loaders
        gc.collect()
    if not path.is_file():
        raise FileNotFoundError("Refiner has no completed best checkpoint. Increase its budget.")
    saved = torch.load(path, weights_only=True, map_location="cpu")
    if (saved["settings"] != V3 or saved["annotation_sha256"] != annotation_sha256
            or saved["smoke_test"] != SMOKE_TEST):
        raise ValueError("Refiner checkpoint provenance mismatch.")
    refiner = v3_refiner(False)
    refiner.load_state_dict(saved["state_dict"], strict=True)
    return refiner.eval()


@torch.inference_mode()
def v3_proposals(path, detector, refiner, confidence):
    result = detector.predict(
        source=str(path), imgsz=V3["image_size"], conf=confidence,
        device=YOLO_DEVICE, agnostic_nms=True, retina_masks=True, verbose=False,
    )[0]
    if result.masks is None:
        return []
    masks = result.masks.data.cpu().numpy()
    if masks.shape[1:] != FULL_SHAPE or not np.isfinite(masks).all():
        raise ValueError("YOLO masks are not finite full-resolution masks.")
    proposals = []
    with Image.open(path) as image:
        for mask in masks:
            coarse = (mask >= 0.5).astype(np.uint8)
            if not coarse.any():
                print("WARNING: YOLO returned an empty mask; omitted:", path.name)
                continue
            bounds = crop_bounds(coarse)
            x = torch.from_numpy(crop_input(image, bounds))[None].to(DEVICE)
            probability = refiner(x).sigmoid()[0, 0].cpu().numpy()
            if not np.isfinite(probability).all():
                raise FloatingPointError("Nonfinite refinement probability.")
            proposals.append(dict(
                coarse=mask_utils.encode(np.asfortranarray(coarse)),
                bounds=bounds, probability=probability,
            ))
    return proposals


def v3_instances(proposals, parameters):
    instances, fallback = [], 0
    for proposal in proposals:
        coarse = proposal["coarse"]
        if parameters["mode"] == "yolo":
            rle = coarse
        else:
            x0, y0, x1, y1 = proposal["bounds"]
            probability = np.asarray(Image.fromarray(proposal["probability"]).resize(
                (x1-x0, y1-y0), Image.Resampling.BILINEAR
            ))
            crop = probability >= parameters["refiner_threshold"]
            if not crop.any():
                # Retain an actual detector prediction; never synthesize an empty-image mask.
                fallback += 1
                rle = coarse
            else:
                full = np.zeros(FULL_SHAPE, dtype=np.uint8)
                full[y0:y1, x0:x1] = crop
                rle = mask_utils.encode(np.asfortranarray(full))
        if mask_utils.area(rle) >= parameters["min_area"]:
            instances.append(rle)
    instances, _ = disjoint_instances(instances, FULL_SHAPE, parameters["min_area"])
    return instances, fallback


def v3_scores(names, cache, parameters):
    rows = []
    for name in names:
        predicted, fallback = v3_instances(cache[name], parameters)
        scores = [local_scores(predicted, gt_rles(r["id"])) for r in records_by_name[name]]
        rows.append(dict(filename=name, refiner_fallbacks=fallback, **{
            key: float(np.mean([score[key] for score in scores]))
            for key in scores[0]
        }))
    return {key: float(np.mean([r[key] for r in rows]))
            for key in rows[0] if key != "filename"}, rows


def v3_calibrate(detector, refiner):
    hashes = {name: hashlib.sha256((OUTPUT_DIR / name).read_bytes()).hexdigest()
              for name in ["yolo_best.pt", "refiner_best.pt"]}
    path = OUTPUT_DIR / "v3_calibration.json"
    if RUN_V3_CALIBRATION:
        names = list(split_names["tune"])
        random.Random(CFG["seed"]).shuffle(names)
        names = names[:V3["tuning_images"]]
        if not names:
            raise ValueError("No tuning images.")
        trials = []
        for confidence in V3["confidences"]:
            cache = {name: v3_proposals(train_lookup[name], detector, refiner, confidence)
                     for name in tqdm(names, desc=f"Tune confidence={confidence}")}
            for mode in ["yolo", "refined"]:
                thresholds = [None] if mode == "yolo" else V3["refiner_thresholds"]
                for threshold, area in itertools.product(thresholds, V3["min_areas"]):
                    parameters = dict(mode=mode, confidence=confidence,
                                      refiner_threshold=threshold, min_area=area)
                    scores, _ = v3_scores(names, cache, parameters)
                    trials.append(dict(parameters=parameters, scores=scores))
            del cache
        best_by_mode = {
            mode: max((row for row in trials if row["parameters"]["mode"] == mode),
                      key=lambda row: row["scores"]["pq"])
            for mode in ["yolo", "refined"]
        }
        selected = max(best_by_mode.values(), key=lambda row: row["scores"]["pq"])
        saved = dict(
            best_by_mode=best_by_mode, selected=selected, candidates=trials,
            filenames=names, hashes=hashes, settings=V3,
            annotation_sha256=annotation_sha256, smoke_test=SMOKE_TEST,
            postprocessing=MASK_POLICY,
        )
        write_json(path, saved)
    else:
        saved = json.loads(path.read_text())
        if (saved.get("postprocessing") != MASK_POLICY
                or saved["hashes"] != hashes or saved["settings"] != V3
                or saved["smoke_test"] != SMOKE_TEST
                or saved["annotation_sha256"] != annotation_sha256):
            raise ValueError("V3 calibration does not match current weights/settings.")
    print("TUNING ONLY, not independent performance:", json.dumps(saved["best_by_mode"], indent=2))
    print("Selected stage:", saved["selected"]["parameters"])
    return saved


def v3_export(detector, refiner, calibration):
    if SMOKE_TEST:
        print("Smoke run: competition export disabled.")
        return None
    if calibration.get("postprocessing") != MASK_POLICY:
        raise ValueError("Re-run calibration with the current disjoint-mask policy; no retraining needed.")
    hashes = {name: hashlib.sha256((OUTPUT_DIR / name).read_bytes()).hexdigest()
              for name in ["yolo_best.pt", "refiner_best.pt"]}
    if (calibration["hashes"] != hashes or calibration["settings"] != V3
            or calibration["annotation_sha256"] != annotation_sha256
            or calibration["smoke_test"] != SMOKE_TEST):
        raise ValueError("Export calibration does not match current checkpoints/settings.")
    parameters = calibration["selected"]["parameters"]
    rows, audit = [], []
    for path in tqdm(test_paths, desc="V3 test instances"):
        proposals = v3_proposals(path, detector, refiner, parameters["confidence"])
        instances, fallback = v3_instances(proposals, parameters)
        validate_disjoint(instances, FULL_SHAPE)
        audit.append(dict(filename=path.name, instances=len(instances), refiner_fallbacks=fallback))
        for index, rle in enumerate(instances, 1):
            mask = mask_utils.decode(rle)
            counts = rle["counts"].decode("ascii")
            if mask.shape != FULL_SHAPE or not mask.any():
                raise ValueError("Invalid V3 full-resolution instance.")
            if mask_utils.encode(np.asfortranarray(mask))["counts"].decode("ascii") != counts:
                raise ValueError("V3 RLE roundtrip failed.")
            rows.append(dict(filament_id=f"{path.stem}_{index}", segmentation_rle=counts))
    zeros = [r["filename"] for r in audit if not r["instances"]]
    write_json(OUTPUT_DIR / "test_audit.json", dict(images=audit, zero_detection_images=zeros))
    if not rows:
        print("NOT READY: no instances predicted; no CSV written.")
        return None
    if len(set(r["filament_id"] for r in rows)) != len(rows):
        raise ValueError("Duplicate V3 IDs.")
    validation = validate_submission_rows(rows, {p.stem for p in test_paths}, FULL_SHAPE)
    path = OUTPUT_DIR / "submission.csv"
    temporary = path.with_suffix(".tmp.csv")
    with temporary.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["filament_id", "segmentation_rle"])
        writer.writeheader(); writer.writerows(rows)
    with temporary.open(newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["filament_id", "segmentation_rle"] or list(reader) != rows:
            raise ValueError("V3 CSV validation failed.")
    temporary.replace(path)
    write_json(OUTPUT_DIR / "submission_manifest.json", dict(
        rows=len(rows), images=len(audit), parameters=parameters,
        hashes=calibration["hashes"], annotation_sha256=annotation_sha256,
        csv_sha256=hashlib.sha256(path.read_bytes()).hexdigest(), smoke_test=False,
        zero_detection_images=zeros, postprocessing=MASK_POLICY, validation=validation,
    ))
    if Path("/kaggle/working").is_dir():
        shutil.copyfile(path, "/kaggle/working/submission.csv")
    if zeros:
        print("WARNING: zero-detection images omitted, no invented masks:", zeros)
    print("CSV PREPARED:", path, "No API upload performed.")
    return path


## EDA: connect the mathematics to real images
The following figures use **training images only**. Class-conditional
histograms use one annotator's target: background includes off-disk pixels
and any filaments that annotator did not mark. Nonzero pixels are not an
accurate solar-disk segmentation. Annotation disagreements are not silently
combined into a consensus.

### Intensities and edges
Original grayscale intensities are $x\in[0,255]$. The crop U-Net sees
$$x'=2(x/255)-1.$$
This affine mapping changes the numeric scale, not the spatial structure.
Ultralytics handles the detector's own input normalization; these are
different model interfaces. Replicating grayscale into three channels does
not add color information.

An intensity histogram estimates how often a brightness occurs:
$$h(k)=\frac{1}{N}\sum_{i=1}^{N}\mathbf{1}[x_i=k].$$
Overlap between filament and background histograms explains why one global
brightness threshold may fail. Spatial context matters.
The displayed edge map uses
$$|\nabla I|=\sqrt{(\partial_x I)^2+(\partial_y I)^2}.$$
It is a finite-difference EDA visualization, **not an extra detector input**.
Edges can belong to filaments, the solar limb, or artifacts.

### Spatial sampling and shapes
Reducing 2048px to 512px gives a factor of four per axis; to 1024px gives a
factor of two. A hypothetical four-pixel-wide filament occupies roughly
one pixel at 512 and two at 1024. This is a sampling illustration, not a
measured recall improvement. Images are bilinearly interpolated; binary
targets use nearest-neighbor sampling to avoid fractional labels.

Foreground fraction is
$$f=\frac{\sum_i y_i}{HW}.$$
The resolution table shows how raster sampling changes this fraction;
it does not quantify model accuracy. Instance area is $A=\sum_i y_i$.
Box fill $A/(wh)$ and elongation $\max(w,h)/\min(w,h)$ describe geometry.
Elongation is a bounding-box proxy, not filament spine length or chirality.

YOLO polygon coordinates are normalized:
$$u=x/W,\qquad v=y/H.$$
A contextual crop is resized to 256px; inference maps its probability
back into the original crop bounds before applying the refiner threshold.
Yellow rectangles in the figure are crop regions, not learned layer shapes.


In [ ]:
# Training-only exploratory sample; no holdout or test-driven parameter selection.
EDA_IMAGE_LIMIT = 24
eda_names = list(split_names["train"])
random.Random(CFG["seed"]).shuffle(eda_names)
eda_names = eda_names[:EDA_IMAGE_LIMIT]
if not eda_names:
    raise ValueError("Training EDA has no images.")
eda_name = eda_names[0]
eda_record = records_by_name[eda_name][0]
with Image.open(train_lookup[eda_name]) as image:
    eda_gray = np.asarray(image).copy()
eda_truth = union_mask(gt_rles(eda_record["id"]))
if not eda_truth.any():
    raise ValueError("Select a training annotation set with foreground for EDA.")

figure, axes = plt.subplots(2, 3, figsize=(15, 9))
axes[0, 0].imshow(eda_gray, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Original 2048px grayscale")
axes[0, 1].imshow(eda_gray.astype(np.float32) / 127.5 - 1, cmap="gray", vmin=-1, vmax=1)
axes[0, 1].set_title("Crop-model normalization: x / 127.5 - 1")
axes[0, 2].imshow(eda_gray, cmap="gray", vmin=0, vmax=255)
axes[0, 2].contour(eda_truth, levels=[0.5], colors=["cyan"], linewidths=0.5)
axes[0, 2].set_title(f"One annotator's union: {eda_record['id']}")
for axis in axes[0]:
    axis.axis("off")
bins = np.arange(257)
axes[1, 0].hist(eda_gray.ravel(), bins=bins, density=True, histtype="step", label="All pixels")
axes[1, 0].hist(eda_gray[eda_truth.astype(bool)], bins=bins, density=True, histtype="step", label="Filament pixels")
axes[1, 0].hist(eda_gray[~eda_truth.astype(bool)], bins=bins, density=True, histtype="step", label="Background + off-disk")
axes[1, 0].set(xlabel="Intensity (0-255)", ylabel="Density", title="Class-conditional intensity overlap")
axes[1, 0].legend(fontsize=8)
axes[1, 1].hist(eda_gray[eda_gray > 0], bins=bins, density=True, histtype="step")
axes[1, 1].set(xlabel="Nonzero intensity", ylabel="Density", title="Nonzero pixels (NOT a validated disk mask)")
gradient_y, gradient_x = np.gradient(eda_gray.astype(np.float32) / 255)
gradient = np.hypot(gradient_x, gradient_y)
axes[1, 2].imshow(gradient, cmap="magma", vmin=0, vmax=float(np.quantile(gradient, .995)))
axes[1, 2].set_title("Finite-difference gradient magnitude")
axes[1, 2].axis("off")
figure.suptitle(f"Training observation: {eda_name}", fontsize=12)
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "eda_intensity_and_targets.png", dpi=120)
plt.show()

# Display geometry at different sizes; these normalized areas are not recall measurements.
sizes = [256, 512, 1024]
figure, axes = plt.subplots(2, 3, figsize=(15, 9))
resolution_rows = []
for column, size in enumerate(sizes):
    observation = np.asarray(Image.fromarray(eda_gray).resize((size, size), Image.Resampling.BILINEAR))
    target = np.asarray(Image.fromarray(eda_truth).resize((size, size), Image.Resampling.NEAREST))
    axes[0, column].imshow(observation, cmap="gray", vmin=0, vmax=255)
    axes[0, column].set_title(f"{size}px input; {2048 / size:g}x downsampling")
    axes[1, column].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[1, column].set_title(f"Nearest target: {int(target.sum())} foreground pixels")
    for axis in axes[:, column]:
        axis.axis("off")
    resolution_rows.append(dict(
        size=size, foreground_pixels=int(target.sum()),
        foreground_fraction=float(target.mean()), original_foreground_fraction=float(eda_truth.mean()),
    ))
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "eda_resolution.png", dpi=120)
plt.show()
write_json(OUTPUT_DIR / "eda_resolution.json", resolution_rows)
print("Spatial sampling changes (not measured segmentation accuracy):", resolution_rows)

instance_rows, image_rows = [], []
for name in eda_names:
    # One deterministic annotation set per image avoids multiplying image weights in the EDA.
    record = records_by_name[name][0]
    rles = gt_rles(record["id"])
    union = union_mask(rles)
    image_rows.append(dict(
        filename=name, annotator_sets=len(records_by_name[name]),
        instances=len(rles), foreground_fraction=float(union.mean()),
    ))
    for annotation, rle in zip(annotations_by_id[record["id"]], rles):
        area = float(mask_utils.area(rle))
        x, y, width, height = mask_utils.toBbox(rle).tolist()
        instance_rows.append(dict(
            filename=name, annotation_id=annotation["id"], area=area,
            bbox_width=width, bbox_height=height,
            elongation=max(width, height) / max(1, min(width, height)),
            bbox_fill=area / max(1, width * height),
        ))
if not instance_rows:
    raise ValueError("No foreground instances in EDA sample.")
figure, axes = plt.subplots(1, 4, figsize=(17, 4))
axes[0].hist([row["instances"] for row in image_rows], bins=12)
axes[0].set(xlabel="Instances / annotation set", title="Instance counts")
axes[1].hist(np.log10([row["area"] for row in instance_rows]), bins=20)
axes[1].set(xlabel="log10 area (original pixels)", title="Filament sizes")
axes[2].hist([row["elongation"] for row in instance_rows], bins=20)
axes[2].set(xlabel="Long / short bbox side", title="Box elongation proxy")
axes[3].hist([row["bbox_fill"] for row in instance_rows], bins=20)
axes[3].set(xlabel="Mask area / box area", title="Irregular shapes vs boxes")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "eda_instance_geometry.png", dpi=120)
plt.show()
write_json(OUTPUT_DIR / "eda_geometry.json", dict(
    policy="training-only sample, one annotation set per physical image",
    images=image_rows, instances=instance_rows,
))

# Follow one instance through normalized YOLO coordinates and the refiner crop.
annotation = annotations_by_id[eda_record["id"]][0]
instance = mask_utils.decode(ann_rle(annotation))
bounds = crop_bounds(instance)
x0, y0, x1, y1 = bounds
with Image.open(train_lookup[eda_name]) as image:
    crop = crop_input(image, bounds)
target = np.asarray(Image.fromarray(instance[y0:y1, x0:x1]).resize(
    (V3["crop_size"], V3["crop_size"]), Image.Resampling.NEAREST
))
polygon = np.asarray(annotation["segmentation"][0]).reshape(-1, 2)
figure, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(eda_gray, cmap="gray", vmin=0, vmax=255)
axes[0].plot(*np.vstack([polygon, polygon[0]]).T, color="cyan", linewidth=1)
axes[0].add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor="yellow"))
axes[0].set_title("Polygon (cyan), context crop (yellow)")
axes[1].imshow(crop[0], cmap="gray", vmin=-1, vmax=1)
axes[1].set_title(f"{V3['crop_size']}px crop; 3 identical channels")
axes[2].imshow(target, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Single-instance target, not union of neighbors")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "eda_crop_path.png", dpi=120)
plt.show()
print("YOLO polygon first coordinates, normalized by 2048:", (polygon / 2048)[:3])
print("Refiner crop bounds in original pixels:", bounds)
print("Training EDA does not change V3 data, labels, or hyperparameters.")


## Grouped supervision and detector training
The V1 fixed observation-day partition is recomputed with seed 2026. All
annotator label sets for a physical image stay together. Each annotation set
gets a separate symlinked image/label pair, avoiding a union of incompatible
annotators or dropping less-complete annotations. YOLO labels represent
individual polygons in normalized coordinates. All categories become
foreground class 0: chirality is not part of the submission task.

Only single-piece polygons are supported for YOLO conversion; incompatible
annotations raise explicit errors rather than silently losing geometry.
This is supported by the current competition annotations, not a general
COCO-to-YOLO converter. Original 2048px images stay on disk; detector resizing
uses Ultralytics letterboxing. Full-resolution retinal masks are required.
Detector checkpoint selection uses Ultralytics validation metrics, followed
by our tuning-only PQ calibration. Ultralytics metrics are not Kaggle PQ.

This V3 uses one grouped split, **not K-fold**. The separate V2 grouped-fold
notebook remains available for a larger-budget study. V1 holdout is not used
again here for selection or purported untouched evaluation.

In [ ]:
detector = v3_train_yolo()

## Instance crop refiner
For each annotation, generate a square crop around its COCO mask with 1.8x
context and a 96px minimum side. Clamp crops within the 2048px observation.
Only that instance is the target, not every neighboring filament.
Targets use nearest-neighbor resizing; images use bilinear interpolation.
Grayscale is replicated into three channels and normalized to [-1,1],
identically during training and inference.

Each crop is resized to 256px. Training uses paired quarter rotations and
horizontal flips, AdamW at 3e-4/weight decay 1e-4, Dice+BCE loss, cosine LR
schedule, and lowest tuning-loss checkpoint selection. CUDA AMP is used for
the network, with float32 loss reductions.

**Limitation:** training crops come from ground-truth instances; test crops
come from YOLO masks. Proposal misalignment is a distribution shift. Nearby
instances can also be included in a crop. The same image-only refiner applied
to overlapping proposals may generate duplicate regions, which PQ penalizes.
YOLO-only vs refined tuning comparison is therefore required, not assumed.
An empty refined crop falls back to its actual YOLO mask and is counted in
the audit; no detection is fabricated for an empty image.

In [ ]:
refiner = v3_train_refiner()

## Compare YOLO-only and YOLO + refinement
Evaluate both on the same seeded up-to-32 physical tuning images, averaging
annotators per image. Search detector confidence {0.15,0.25}, original
pixel minimum area {20,64}, and refiner probability threshold {0.5,0.65}.
Choose stage and thresholds by tuning PQ only. Report stage-specific PQ,
Dice, IoU, split/merge relations, and fallback counts.

These are parameter-selection metrics, not independent generalization
estimates. No K-fold averages or SD are claimed. The detector/refiner losses
and Ultralytics mAP do not substitute for Kaggle PQ.
Calibration caches compact coarse RLE and crop probabilities one confidence
at a time; no huge cache of full 2048px masks for all trials.
Ties choose YOLO-only by stable ordering, avoiding unsupported complexity.

In [ ]:
v3_calibration = v3_calibrate(detector, refiner)

## After training: pixel errors versus instance errors
The crop U-Net produces logits $z$ and probabilities
$$p=\sigma(z)=\frac{1}{1+\exp(-z)}.$$
Its training loss is
$$\mathcal{L}=\frac12\mathcal{L}_{\mathrm{BCE}}+
\frac12\left(1-\frac{2\sum_i p_i y_i+1}{\sum_i p_i+\sum_i y_i+1}\right),$$
where
$$\mathcal{L}_{\mathrm{BCE}}=-\frac1N\sum_i\left[y_i\log p_i+
(1-y_i)\log(1-p_i)\right].$$
The implementation computes BCE from logits for numerical stability.
These equations describe our **crop refiner loss**, not Ultralytics'
entire detector loss.

YOLO's object confidence threshold and the refiner's pixel probability
threshold are **not interchangeable**. Confidence determines which instance
proposals survive; the pixel threshold determines the mask inside each crop.
The area filter operates in **original 2048px pixels** after remapping.

For binary masks,
$$\mathrm{Dice}=\frac{2|P\cap G|}{|P|+|G|},\qquad
\mathrm{IoU}=\frac{|P\cap G|}{|P\cup G|}.$$
The colored overlay below highlights true-positive pixels (green),
false negatives (yellow), and false positives (red).
Those are **pixel errors**; connected shapes are not automatically
correct instances. PQ measures one-to-one instance matches:
$$\mathrm{PQ}=
\frac{\sum_{(p,g)\in\mathrm{TP}}\mathrm{IoU}(p,g)}
{|\mathrm{TP}|+\frac12|\mathrm{FP}|+\frac12|\mathrm{FN}|}.$$
Matching requires mask IoU **strictly greater than 0.5**.
The heatmap shows pairwise overlaps before matching. A broad mask spanning
two human instances can have many correct foreground pixels but low PQ;
duplicate refined masks can similarly add false-positive instances.

This visual review uses a selected tuning image and one annotator. It is
qualitative debugging, not a fresh independent evaluation or Kaggle score.


In [ ]:
# Qualitative tuning review only; this is not an independent test metric.
review_name = v3_calibration["filenames"][0]
parameters = v3_calibration["selected"]["parameters"]
proposals = v3_proposals(train_lookup[review_name], detector, refiner, parameters["confidence"])
predicted, fallback_count = v3_instances(proposals, parameters)
record = records_by_name[review_name][0]
truth_rles = gt_rles(record["id"])
truth = union_mask(truth_rles).astype(bool)
prediction = union_mask(predicted).astype(bool)
with Image.open(train_lookup[review_name]) as image:
    observation = np.asarray(image)
error = np.repeat((observation.astype(np.float32) / 255)[..., None], 3, axis=2)
error[truth & prediction] = [0, 1, 0]
error[truth & ~prediction] = [1, 1, 0]
error[~truth & prediction] = [1, 0, 0]
figure, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(observation, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Tuning observation")
axes[1].imshow(observation, cmap="gray", vmin=0, vmax=255)
if prediction.any():
    axes[1].contour(prediction, levels=[.5], colors=["cyan"], linewidths=.6)
axes[1].set_title(f"Selected {parameters['mode']}: {len(predicted)} instances")
axes[2].imshow(error)
axes[2].set_title("Green: TP | yellow: FN | red: FP pixels")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "tuning_error_review.png", dpi=120)
plt.show()
scores = local_scores(predicted, truth_rles)
print("ONE tuning image / ONE annotator (not aggregate):", review_name, record["id"], scores)
print("Audited empty-refinement fallbacks:", fallback_count)
if predicted and truth_rles:
    overlaps = mask_utils.iou(predicted, truth_rles, [0] * len(truth_rles))
    figure, axis = plt.subplots(figsize=(7, 5))
    artist = axis.imshow(overlaps, vmin=0, vmax=1, cmap="magma")
    axis.set(xlabel="Human instance index", ylabel="Predicted instance index",
             title="Pairwise mask IoU: PQ requires > 0.5")
    figure.colorbar(artist, ax=axis, label="IoU")
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "tuning_instance_iou.png", dpi=120)
    plt.show()
else:
    print("No pairwise IoU heatmap: one side has no instances.")


## Manual competition export
Only the tuning-selected mode is exported. Each predicted instance has a
2048x2048 nonempty mask; RLE roundtrip, CSV readback, hashes, unique IDs, and
zero overlapping pixels within each image are checked. Zero-detection images
have no rows and are reported. V1/V2
submissions accepted omitted rows; acceptance of V3 is still Kaggle's decision.
No fabricated masks or per-test-image threshold changes.

After `CSV PREPARED`, save the finished Kaggle notebook version and select
`/kaggle/working/submission.csv` using Submit, or download that file and upload
it manually. Training does **not** trigger a competition submission.
Download the entire experiment directory before resetting the session.
Set both V3 training flags False to reuse saved checkpoints.
Keep RUN_V3_CALIBRATION=True when upgrading from the original overlap-permitting
export. Only reuse calibration with the matching mask policy.
Use a new RUN_NAME for changed training settings.

In [ ]:
if RUN_V3_EXPORT:
    v3_submission_path = v3_export(detector, refiner, v3_calibration)
else:
    print("V3 test export disabled. No competition upload performed.")


In [ ]:
with (OUTPUT_DIR / "requirements-actual.txt").open("w") as stream:
    subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                   stdout=stream, check=True)
print("Save the full evidence directory:", OUTPUT_DIR)
print("Weights: yolo_best.pt and refiner_best.pt; configuration: v3_plan.json")
print("Selection metrics: v3_calibration.json. Historical offline-repaired V3 public PQ: 0.35; not a score for new runs.")


## References and provenance
- [Ultralytics segmentation documentation](https://docs.ultralytics.com/tasks/segment/) and [licensing](https://www.ultralytics.com/license).
- Ronneberger et al. (2015), U-Net, doi:10.1007/978-3-319-24574-4_28.
- He et al. (2016), ResNet, doi:10.1109/CVPR.2016.90.
- Kirillov et al. (2019), PQ, doi:10.1109/CVPR.2019.00963.
- Ahmadzadeh et al. (2024), MAGFiLO, doi:10.1038/s41597-024-03876-y.
- Design inspiration: supplied `yolo-u-net-solar-filament-segmentation.ipynb`.
  Public author/URL and reported ~0.69 score have not been independently verified.
  Add the exact public citation before publishing a comparison with that work.

Original implementation only; no third-party reference notebook is modified.
Actual package versions, selected weights, grouping and calibration are
saved with the run. Higher score is a hypothesis, not a guarantee.